In [ ]:
# Установка зависимостей

!pip install openai llama_index llama-index-postprocessor-longllmlingua llmlingua
!pip install kagglehub[pandas-datasets]

In [ ]:
# Импорты

import os
import getpass
import pandas as pd

# KAGGLE DATASET

import kagglehub
from kagglehub import KaggleDatasetAdapter

# LLAMAINDEX

from llama_index.core import (
    VectorStoreIndex,
    Document,
    Settings
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.postprocessor import LLMRerank, LongContextReorder
from llama_index.postprocessor.longllmlingua import LongLLMLinguaPostprocessor

In [ ]:
# API KEY (OpenAI)

os.environ["API_KEY"] = getpass.getpass("Введите OpenAI API Key: ")

Введите OpenAI API Key: ··········


In [ ]:
# Промт — Нейро-Сотрудник

SYSTEM_PROMPT = """
Ты — аналитик данных (стажёр).
Ты отвечаешь ТОЛЬКО на основе предоставленных данных.
Если в данных нет ответа — скажи "Недостаточно данных".
Запрещено:
- использовать внешние знания
- додумывать факты
- делать прогнозы без данных
Каждый вывод должен опираться на контекст.
"""

In [ ]:
# Настройки LLM

Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    system_prompt=SYSTEM_PROMPT
)

Settings.embed_model = OpenAIEmbedding()
Settings.chunk_size = 512

In [ ]:
# Скачивание Dataset
# для примера был взят датасет (https://www.kaggle.com/datasets/muqaddasejaz/ethereum-usd-dataset)

print("Скачивание датасета Ethereum-USD...")

dataset_path = kagglehub.dataset_download(
    "muqaddasejaz/ethereum-usd-dataset"
)

print("Датасет скачан в:", dataset_path)

print("\nФайлы в датасете:")
for file in os.listdir(dataset_path):
    print(" -", file)

Скачивание датасета Ethereum-USD...


100%|██████████| 82.9k/82.9k [00:00<00:00, 54.9MB/s]

Extracting files...
Датасет скачан в: /root/.cache/kagglehub/datasets/muqaddasejaz/ethereum-usd-dataset/versions/1

Файлы в датасете:
 - eth_usd_dataset.csv


In [ ]:
# Загрузка CSV В DataFrame

csv_file = "eth_usd_dataset.csv"
csv_path = os.path.join(dataset_path, csv_file)

df = pd.read_csv(csv_path)

print("\nDataFrame успешно загружен")
print(type(df))
print(df.head())


DataFrame успешно загружен
<class 'pandas.core.frame.DataFrame'>
         Date               Close                High                 Low  \
0         NaN             ETH-USD             ETH-USD             ETH-USD   
1  2017-11-09   320.8840026855469   329.4519958496094   307.0559997558594   
2  2017-11-10  299.25299072265625   324.7179870605469      294.5419921875   
3  2017-11-11   314.6809997558594   319.4530029296875   298.1919860839844   
4  2017-11-12   307.9079895019531  319.15301513671875  298.51300048828125   

                 Open      Volume  
0             ETH-USD     ETH-USD  
1   308.6449890136719   893249984  
2   320.6709899902344   885985984  
3  298.58599853515625   842300992  
4  314.69000244140625  1613479936  


In [ ]:
# Преобразование Dataset в Document

documents = []

for idx, row in df.iterrows():
    text = (
        f"Дата: {row['Date']}\n"
        f"Цена открытия: {row['Open']}\n"
        f"Максимум: {row['High']}\n"
        f"Минимум: {row['Low']}\n"
        f"Цена закрытия: {row['Close']}\n"
        f"Объём: {row['Volume']}"
    )

    doc = Document(
        text=text,
        metadata={
            "source": "ethereum-usd-dataset",
            "row_id": idx,
            "type": "financial_timeseries"
        }
    )
    documents.append(doc)

print(f"Создано документов: {len(documents)}")

Создано документов: 2610


In [ ]:
# Ссоздание index

index = VectorStoreIndex.from_documents(documents)

In [ ]:
# Фильтрация запросов

def is_safe_query(query: str) -> bool:
    blacklist = [
        "ignore previous",
        "system prompt",
        "jailbreak",
        "придумай",
        "ответь без данных"
    ]
    return not any(word in query.lower() for word in blacklist)

In [ ]:
# RAG

query_engine = index.as_query_engine(
    similarity_top_k=20,
    node_postprocessors=[
        LLMRerank(top_n=5),
        LongContextReorder(),

    ]
)

In [ ]:
# Трассировка и тестовые запросы

test_queries = [
    "Какая была цена закрытия Ethereum в самый последний день?",
    "Какой был максимальный High в датасете?",
    "Есть ли данные о цене в 2030 году?"
]

for query in test_queries:
    print("\n" + "=" * 80)
    print(f"ВОПРОС: {query}")

    if not is_safe_query(query):
        print("Запрос отклонён по соображениям безопасности")
        continue

    response = query_engine.query(query)

    print("\nОТВЕТ НЕЙРО-СОТРУДНИКА:")
    print(response)


    # Ттрассровка источников

    print("\nИСТОЧНИКИ (source nodes):")
    if not response.source_nodes:
        print("❌ Источники не использованы — возможна галлюцинация")
    else:
        for i, node in enumerate(response.source_nodes, 1):
            print(f"\n--- SOURCE NODE {i} ---")
            print("METADATA:", node.node.metadata)
            print("TEXT (fragment):")
            print(node.node.text[:300])


ВОПРОС: Какая была цена закрытия Ethereum в самый последний день?

ОТВЕТ НЕЙРО-СОТРУДНИКА:
Цена закрытия Ethereum в самый последний день (2024-05-13) составила 2949.359619140625.

ИСТОЧНИКИ (source nodes):

--- SOURCE NODE 1 ---
METADATA: {'source': 'ethereum-usd-dataset', 'row_id': 2378, 'type': 'financial_timeseries'}
TEXT (fragment):
Дата: 2024-05-13
Цена открытия: 2928.81396484375
Максимум: 2994.869140625
Минимум: 2865.134521484375
Цена закрытия: 2949.359619140625
Объём: 13352264795

--- SOURCE NODE 2 ---
METADATA: {'source': 'ethereum-usd-dataset', 'row_id': 2015, 'type': 'financial_timeseries'}
TEXT (fragment):
Дата: 2023-05-16
Цена открытия: 1816.82421875
Максимум: 1830.3515625
Минимум: 1797.84375
Цена закрытия: 1824.1214599609375
Объём: 5595959668

ВОПРОС: Какой был максимальный High в датасете?

ОТВЕТ НЕЙРО-СОТРУДНИКА:
Максимальный High в датасете составил 3534.826416015625.

ИСТОЧНИКИ (source nodes):

--- SOURCE NODE 1 ---
METADATA: {'source': 'ethereum-usd-dataset', 'row_id